In [1]:
import os
import pandas as pd
import spacy
import nltk
from nltk.corpus import wordnet as wn

In [2]:

nlp = spacy.load("en_core_web_lg")

In [3]:

collPath = "s2"

In [4]:
for file in os.listdir(collPath):
    if file.endswith(".txt"):
        filepath = f"{collPath}/{file}"
        name, extension = os.path.splitext(file)

        with open(filepath, "r", encoding="utf8") as f:
            readFile = f.read()
            print(name, "----", len(readFile), "characters")

02x01_-_Follow_Mama_and_Papa ---- 12558 characters
02x02_-_Bond_s_Strategy_to_Stay_Alive-Damian_s_Field_Research_Trip ---- 11470 characters
02x03_-_Mission_and_Family-The_Elegant_Bondman-The_Heart_of_a_Child ---- 10821 characters
02x04_-_The_Pastry_of_Knowledge-The_Informant_s_Great_Romance_Plan_II ---- 13800 characters
02x05_-_Plan_to_Cross_the_Border ---- 14583 characters
02x06_-_The_Fearsome_Luxury_Cruise_Ship ---- 14389 characters
02x07_-_Who_Is_This_Mission_For_ ---- 13177 characters
02x08_-_The_Symphony_Upon_the_Ship-Sis_s_Herb_Tea ---- 10944 characters
02x09_-_The_Hand_That_Connects_to_the_Future ---- 12074 characters
02x10_-_Enjoy_the_Resort_to_the_Fullest-Bragging_About_Vacation ---- 10607 characters
02x11_-_Berlint_in_Love-Nightfall_s_Daily_Life ---- 13780 characters
02x12_-_Part_of_the_Family ---- 10227 characters


In [5]:
def wordCollector(words, unit, pos_choice="ADJ"):
    wordList = []
    nodeTypes = []
    synsetCounts = []
    unitList = []

    for token in words:
        if (
            token.pos_ == pos_choice
            and token.is_alpha
            and not token.is_stop
        ):
            lemma = token.lemma_.lower()
            synsets = len(wn.synsets(lemma))

            wordList.append(lemma)
            nodeTypes.append(token.pos_)
            synsetCounts.append(synsets)
            unitList.append(unit)

    data = {
        "word": wordList,
        "nodeType": nodeTypes,
        "synsetCount": synsetCounts,
        "unit": unitList
    }

    df = pd.DataFrame(data)

    df = (
        df.groupby(["word", "nodeType", "synsetCount", "unit"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
    )

    return df

In [6]:
allDataFrames = []

for file in os.listdir(collPath):
    if file.endswith(".txt"):
        filepath = f"{collPath}/{file}"
        name, extension = os.path.splitext(file)

        with open(filepath, "r", encoding="utf8") as f:
            readFile = f.read()
            spacyRead = nlp(readFile)

            myDataFrame = wordCollector(spacyRead, name, pos_choice="ADJ")
            allDataFrames.append(myDataFrame)

fullDataFrame = pd.concat(allDataFrames, ignore_index=True)

fullDataFrame

,word,nodeType,synsetCount,unit,count
0,able,ADJ,4,02x01_-_Follow_Mama_and_Papa,1
1,actual,ADJ,5,02x01_-_Follow_Mama_and_Papa,1
2,armed,ADJ,5,02x01_-_Follow_Mama_and_Papa,1
3,bad,ADJ,17,02x01_-_Follow_Mama_and_Papa,4
4,bored,ADJ,4,02x01_-_Follow_Mama_and_Papa,1
...,...,...,...,...,...
957,weird,ADJ,3,02x12_-_Part_of_the_Family,1
958,wet,ADJ,9,02x12_-_Part_of_the_Family,4
959,wilted,ADJ,3,02x12_-_Part_of_the_Family,1
960,worth,ADJ,5,02x12_-_Part_of_the_Family,1


In [7]:
outputFilePath = "spy-family-s2-adjectives-network-data.tsv"

fullDataFrame.to_csv(outputFilePath, sep="\t", index=False)

print("Saved TSV file:", outputFilePath)

Saved TSV file: spy-family-s2-adjectives-network-data.tsv
